> **Historical review artifact — not runnable against the current API.** This notebook records an earlier overhaul review, including obsolete constructors, label rules, and stored outputs. Use the maintained tutorials in `docs/tutorials/` for current examples. Refreshing this tour requires migrating the examples and re-executing every section, including the optional inference backends.
>
> **AI-generated notice:** Codex added this status notice; the historical review below is preserved as evidence.
> Follow-up: [refresh and re-execute the tour](https://github.com/TARPS-group/prob-pipe/issues/528).


# A tour of the rebuilt ProbPipe API

This notebook tours the API on `dev/overhaul`, and its running example is the non-centered eight-schools model, whose parameters form the nested record `{population: {mu, tau}, groups: {theta_tilde}}`.
Each markdown cell says what the next cell shows, and each output is the library's own repr, report, or error message, or a table built from them.
A cell marked **Bug** shows a behavior that departs from the design reference in `design/` and names the section it departs from.
The last section lists what felt awkward while writing the notebook.

## Status

The items below are transitional, and a cell that shows one is marked **Transitional**.

1. **`cov`** returns a dense array for now, and a later change returns a linear operator over the flattened draw.
2. **Evaluation rules.** The evaluation-rule registry holds three rules for now: the empirical enumeration, the elementwise sweep, and the sampling lift. The exact rules for closed forms, changes of variables, and linear maps come in later changes, so `evaluate` of a parametric law takes the sampling lift.
3. **Record arithmetic.** `+`, `-`, and scalar `*` and `/` on a `NumericRecord` raise `TypeError` until the vector-space operators are implemented, and record arithmetic goes through `map` and `to_vector` until then.
4. **Monte Carlo marginals.** `marginal` has no Monte Carlo route until the empirical marginal of projected draws is implemented, so a field with no exact marginal raises `ResolutionError`.
5. **Docs.** The other notebooks in `docs/` use the earlier implementations until a later pass rewrites them.

## Setup

`show` prints several properties of one object as an aligned block, and `spec_table` tabulates the leaves of a declaration.

In [1]:
import tempfile
import warnings
from pathlib import Path
from typing import Any

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

import probpipe
from probpipe import (
    BijectorTransformedDistribution, BootstrapDistribution, BootstrapReplicateDistribution,
    Dirichlet, DistributionBatch, EmpiricalDistribution, Gamma, HalfCauchy, KDEDistribution,
    MathematicalDomainError, MultivariateNormal, Normal, NumericArray, NumericArrayBatch,
    NumericArraySpec, NumericRecordBatch, Opaque, OutputSpec, Poisson, PyMCModel, Record,
    RecordBatch, ResolutionError, StanModel, StudentT, SupportsCovariance,
    SupportsLogProb, SupportsMean, SupportsQuantile, SupportsSampling,
    SupportsUnnormalizedLogProb, SupportsVariance, bijector_for, condition_on,
    conditional_distribution, convert, converter_registry, cov, evaluate, expectation, factor,
    function, iterate, log_prob, marginal, mean, positive, provenance_ancestors, quantile,
    real, replay_run, sample, unnormalized_log_prob, variance, workflow_run,
)
from probpipe.distributions import SupportsFactors, SupportsMarginals
from probpipe.families import PoissonFamily, glm_likelihood
from probpipe.operations import operation_registry

warnings.filterwarnings("ignore", message=r"Explicitly requested dtype float64")
warnings.filterwarnings("ignore", message=r"Loading a shared object")
pd.set_option("display.max_colwidth", None)


def show(properties):
    """Print each property on a line of its own, with the names aligned in one column."""
    width = max(map(len, properties))
    for name, value in properties.items():
        text = str(value).replace("\n", "\n" + " " * (width + 2))
        print(f"{name:<{width}}  {text}")


def spec_table(specs):
    """The shape, dtype, and support of each leaf of *specs*, a record spec or a mapping of specs."""
    leaves = {path: specs[path] for path in specs}
    return pd.DataFrame(
        {"shape": [spec.shape for spec in leaves.values()],
         "dtype": [spec.dtype for spec in leaves.values()],
         "support": [spec.support for spec in leaves.values()]},
        index=list(leaves),
    )

## 1. Values

A value is a tracked term: a representation together with a label and a spec.
`NumericArray` holds one array, whose shape is the event shape, and arithmetic on it returns a tracked term labeled by the expression.

In [2]:
effect = NumericArray("effect", jnp.array([28.0, 8.0, -3.0]))
show({"effect": repr(effect), "effect.spec": effect.spec,
      "effect + 1.0": repr(effect + 1.0), "2 * effect": repr(2 * effect)})

effect        NumericArray('effect', shape=(3,), dtype=float32)
effect.spec   NumericArraySpec(shape=(3,), dtype=float32)
effect + 1.0  NumericArray('effect + 1.0', shape=(3,), dtype=float32)
2 * effect    NumericArray('2 * effect', shape=(3,), dtype=float32)


A `Record` is a named tree of values, and a mapping-valued field becomes a subtree.
A subtree whose leaves are all numeric is promoted to a `NumericRecord`, so a school's record holds a numeric `data` subtree beside a string label.

In [3]:
school = Record("school_A", {"data": {"effect": 28.0, "se": 15.0}, "label": "A"})
show({"school": repr(school), "school.spec": school.spec,
      'school.at_path("data")': repr(school.at_path("data"))})

school                  Record('school_A', fields=('data/effect', 'data/se', 'label'))
school.spec             RecordSpec(data=NumericRecordSpec(effect=(), se=()), label=OpaqueSpec(type=str))
school.at_path("data")  NumericRecord('data', fields=('effect', 'se'))


Indexing a record by a key returns a tracked view of the field's kind, and the view is labeled by the key.

In [4]:
show({'school["data/effect"]': repr(school["data/effect"]), 'school["label"]': repr(school["label"])})

school["data/effect"]  NumericArray('data/effect', shape=(), dtype=float32)
school["label"]        Opaque('label', type=str)


The eight-schools parameters form a `NumericRecord`.
It has the flat-vector interface, and `map` applies an elementwise function and keeps the structure.

In [5]:
theta = Record("theta", {"population": {"mu": 4.0, "tau": 3.0},
                         "groups": {"theta_tilde": jnp.linspace(-1.0, 1.0, 8)}})
show({"theta": repr(theta), "theta.vector_size": theta.vector_size,
      "theta.to_vector()": np.asarray(theta.to_vector()).round(2),
      "theta.map(jnp.abs)": repr(theta.map(jnp.abs))})

theta               NumericRecord('theta', fields=('population/mu', 'population/tau', 'groups/theta_tilde'))
theta.vector_size   10
theta.to_vector()   [ 4.    3.   -1.   -0.71 -0.43 -0.14  0.14  0.43  0.71  1.  ]
theta.map(jnp.abs)  NumericRecord('theta', fields=('population/mu', 'population/tau', 'groups/theta_tilde'))


**Transitional:** record arithmetic raises `TypeError` (Status).

In [6]:
try:
    theta + theta
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

TypeError: unsupported operand type(s) for +: 'NumericRecord' and 'NumericRecord'


`Opaque` wraps a value that no other kind admits, such as a tuple, and its spec records the value's type.

In [7]:
note = Opaque("source", ("Rubin (1981)", "SAT points"))
show({"note": repr(note), "note.spec": note.spec, "note.value": note.value})

note        Opaque('source', type=tuple)
note.spec   OpaqueSpec(type=tuple)
note.value  ('Rubin (1981)', 'SAT points')


A mapping is refused, since the value layer reads a mapping as a subtree.

In [8]:
try:
    Opaque("source", {"study": "Rubin (1981)"})
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

TypeError: Opaque holds one unstructured value, and the value layer reads a mapping as a subtree rather than a leaf; wrap it as a Record, or as a non-mapping value


A `RecordBatch` stores records column by column on named levels.
Indexing by a position gives a record.

In [9]:
y_obs = jnp.array([28.0, 8.0, -3.0, 7.0, -1.0, 1.0, 18.0, 12.0])
sigma = jnp.array([15.0, 10.0, 16.0, 11.0, 9.0, 11.0, 10.0, 18.0])
J = y_obs.shape[0]
schools_data = RecordBatch.stack(
    [Record("school", {"effect": y, "se": s, "label": label})
     for y, s, label in zip(y_obs, sigma, "ABCDEFGH")],
    level_name="school",
    label="schools",
)
show({"schools_data": repr(schools_data), "schools_data.element_spec": schools_data.element_spec,
      "schools_data[2]": repr(schools_data[2]),
      "schools_data.at_levels(school=2)": repr(schools_data.at_levels(school=2))})

schools_data                      RecordBatch('schools', levels={'school': 8}, fields=('effect', 'se', 'label'))
schools_data.element_spec         RecordSpec(effect=(), se=(), label=OpaqueSpec(type=str))
schools_data[2]                   Record('schools[school=2]', fields=('effect', 'se', 'label'))
schools_data.at_levels(school=2)  Record('schools[school=2]', fields=('effect', 'se', 'label'))


Indexing by a field gives its column as a batch of the field's kind: an `OpaqueBatch` for the labels and a `NumericArrayBatch` for the effects.

In [10]:
show({'schools_data["label"]': repr(schools_data["label"]),
      'schools_data["effect"]': repr(schools_data["effect"])})

schools_data["label"]   OpaqueBatch('label', levels={'school': 8}, element_spec=OpaqueSpec(type=str))
schools_data["effect"]  NumericArrayBatch('effect', levels={'school': 8}, element_spec=NumericArraySpec(shape=()))


Every kind defines `raw()`, which returns its representation detached from the workflow.
A value's raw form is its array, its nested mapping, or its stored object.
A parametric law's raw form is its backend object, and an empirical law's is the law itself, detached.

In [11]:
def type_name(raw):
    if isinstance(raw, jax.Array):
        return "jax.Array"
    package = type(raw).__module__.partition(".")[0]
    return type(raw).__name__ if package == "builtins" else f"{package}.{type(raw).__name__}"


terms = [effect, school, note, schools_data, schools_data["effect"],
         EmpiricalDistribution("effect", y_obs), Normal("x", 0.0, 1.0)]
pd.DataFrame({"type of raw()": [type_name(term.raw()) for term in terms]},
             index=pd.Index([type(term).__name__ for term in terms], name="kind"))

,type of raw()
kind,
NumericArray,jax.Array
Record,dict
Opaque,tuple
RecordBatch,dict
NumericArrayBatch,jax.Array
EmpiricalDistribution,probpipe.EmpiricalDistribution
Normal,tensorflow_probability.Normal


## 2. Naming

A label names an object and has no mathematical meaning, while a component name addresses a part of what the object produces.
A family's component defaults to its label, and `event_spec` names another.

In [12]:
prior_beta = Normal("prior", 0.0, 1.0, event_spec=OutputSpec(beta=None))
prior_beta

Normal(
    'prior',
    loc=0.0,
    scale=1.0,
    event_spec=OutputSpec(beta=NumericArraySpec(shape=(), dtype=float32, support=real)),
)

`with_label` changes the label alone, and indexing a whole-term law by its component returns the law itself.

In [13]:
renamed = prior_beta.with_label("slope_prior")
show({"label": prior_beta.label, "components": list(prior_beta.event_spec.components),
      "label after with_label": renamed.label,
      "components after with_label": list(renamed.event_spec.components),
      'prior_beta["beta"] is prior_beta': prior_beta["beta"] is prior_beta})

label                             prior
components                        ['beta']
label after with_label            slope_prior
components after with_label       ['beta']
prior_beta["beta"] is prior_beta  True


`with_path_names` renames nodes by their exact paths.
A target under the same parent renames in place, and a target under another parent moves the node, which promotes or demotes it.
All renames apply at once, so a swap is one call.

In [14]:
swapped = school.with_path_names({"data/effect": "data/se", "data/se": "data/effect"})
show({"in place": list(school.with_path_names({"data/effect": "data/y"}).keys()),
      "promotion": list(school.with_path_names({"data/effect": "effect"}).keys()),
      "demotion": list(school.with_path_names({"label": "meta/label"}).keys()),
      "swap": f'data/effect = {swapped["data/effect"]}, data/se = {swapped["data/se"]}'})

in place   ['data/y', 'data/se', 'label']
promotion  ['data/se', 'label', 'effect']
demotion   ['data/effect', 'data/se', 'meta/label']
swap       data/effect = 15.0, data/se = 28.0


A key must be the exact path of a node, so the bare `effect` is refused, and the `KeyError` lists the paths.

In [15]:
try:
    school.with_path_names({"effect": "y"})
except KeyError as error:
    print(f"{type(error).__name__}: {error}")

KeyError: "with_path_names(): 'effect' is not the path of a node; the paths are ['data', 'data/effect', 'data/se', 'label']"


The same call renames a law's event.
Demoting the fields of a joint of three parametric priors builds the eight-schools prior.
A rename that gathers the components of several factors under one node regroups those factors into a packaged sub-joint, so the renamed prior stays factored.

In [16]:
flat_prior = Normal("mu", 0.0, 5.0) * HalfCauchy("tau", 0.0, 5.0) * Normal("theta_tilde", jnp.zeros(J), 1.0)
NEST = {"mu": "population/mu", "tau": "population/tau", "theta_tilde": "groups/theta_tilde"}
prior = flat_prior.with_path_names(NEST).with_label("prior")
prior

FactoredDistribution(
    'prior',
    factors=(
        FactoredDistribution(
            'mu·tau',
            factors=(Normal('mu', loc=0.0, scale=5.0), HalfCauchy('tau', loc=0.0, scale=5.0)),
            event_spec=OutputSpec(
                population=NumericRecordSpec(
                    mu=NumericArraySpec(shape=(), dtype=float32, support=real),
                    tau=NumericArraySpec(shape=(), dtype=float32, support=greater_than(0.0)),
                ),
            ),
        ),
        FactoredMultivariateGaussian(
            'theta_tilde',
            factors=(
                Normal('theta_tilde', loc=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], scale=1.0),
            ),
            event_spec=OutputSpec(
                groups=NumericRecordSpec(
                    theta_tilde=NumericArraySpec(shape=(8,), dtype=float32, support=real),
                ),
            ),
        ),
    ),
)

Each gathering node is one factor, so the renamed prior has two factors and keeps `SupportsFactors`.

In [17]:
joints = {"flat_prior": flat_prior, "prior": prior}
pd.DataFrame({"components": [list(d.event_spec.components) for d in joints.values()],
              "factors": [[part.label for part in d.factors] for d in joints.values()],
              "SupportsFactors": [isinstance(d, SupportsFactors) for d in joints.values()]},
             index=list(joints))

,components,factors,SupportsFactors
flat_prior,"[mu, tau, theta_tilde]","[mu, tau, theta_tilde]",True
prior,"[population, groups]","[mu·tau, theta_tilde]",True


Demoting the component of a whole-term law is refused, since it would change the draw's kind.

In [18]:
try:
    Normal("mu", 0.0, 5.0).with_path_names({"mu": "population/mu"})
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

ValueError: with_path_names() keeps the packaging, so the whole term's component 'mu' is renamed in place, not moved to 'population/mu'


## 3. Distributions

Each family derives its event declaration from its parameters: the component, shape, dtype, and support.
Capabilities are protocols, so `isinstance(d, SupportsMean)` reports whether a law implements a mean.

In [19]:
families = [
    Normal("mu", 0.0, 5.0),
    HalfCauchy("tau", 0.0, 5.0),
    Poisson("count", 3.0),
    Dirichlet("weights", jnp.ones(3)),
    MultivariateNormal("z", jnp.zeros(2), cov=jnp.array([[1.0, 0.5], [0.5, 2.0]])),
]
protocols = [SupportsSampling, SupportsLogProb, SupportsMean, SupportsVariance, SupportsCovariance,
             SupportsQuantile, SupportsMarginals]
pd.DataFrame(
    {
        "component": [next(iter(d.event_spec.components)) for d in families],
        "shape": [d.event_shape for d in families],
        "dtype": [str(d.dtype) for d in families],
        "support": [str(d.support) for d in families],
        **{p.__name__.removeprefix("Supports"): [isinstance(d, p) for d in families] for p in protocols},
    },
    index=[type(d).__name__ for d in families],
)

,component,shape,dtype,support,Sampling,LogProb,Mean,Variance,Covariance,Quantile,Marginals
Normal,mu,(),float32,real,True,True,True,True,True,True,False
HalfCauchy,tau,(),float32,greater_than(0.0),True,True,True,True,True,True,False
Poisson,count,(),float32,non_negative_integer,True,True,True,True,True,False,False
Dirichlet,weights,"(3,)",float32,simplex,True,True,True,True,True,False,False
MultivariateNormal,z,"(2,)",float32,real,True,True,True,True,True,True,False


A class claims a capability for its instances, and a guard decides whether the capability supports a call on one instance.
`StudentT` claims `SupportsMean`, and its guard admits the call once the degrees of freedom are concrete.
A mean that is undefined raises `MathematicalDomainError`, as a Student t's does at one degree of freedom or fewer, and a mean that diverges is `inf`, as the half-Cauchy's is.

In [20]:
t = StudentT("t", 0.8, 0.0, 1.0)
print("t claims SupportsMean:", isinstance(t, SupportsMean))
try:
    mean(t)
except MathematicalDomainError as error:
    print(f"{type(error).__name__}: {error}")
print("half-Cauchy mean:", mean(HalfCauchy("tau", 0.0, 5.0)))

t claims SupportsMean: True
MathematicalDomainError: the mean of 't' does not exist: it is defined only for degrees of freedom above one
half-Cauchy mean: inf


Parameters with more axes than one law needs give one law whose extra axes are event axes.
A vector `loc` gives independent coordinates, and a `MultivariateNormal` whose `loc` has shape `(4, 2)` gives four independent rows, so `log_prob` of one draw is a scalar in both cases.
Separate laws form a `DistributionBatch` (Section 8).

In [21]:
coords = Normal("theta_tilde", jnp.zeros(J), 1.0)
rows = MultivariateNormal("rows", jnp.zeros((4, 2)), cov=jnp.array([[1.0, 0.8], [0.8, 1.0]]))
with workflow_run(seed=0):
    draw = sample(rows)
show({"coords event": coords.event_spec.spec, "rows event": rows.event_spec.spec,
      "log_prob(coords, 0)": repr(log_prob(coords, jnp.zeros(J))),
      "log_prob(rows, draw)": repr(log_prob(rows, draw))})

coords event          NumericArraySpec(shape=(8,), dtype=float32, support=real)
rows event            NumericArraySpec(shape=(4, 2), dtype=float32, support=real)
log_prob(coords, 0)   NumericArray('theta_tilde', shape=(), dtype=float32)
log_prob(rows, draw)  NumericArray('rows', shape=(), dtype=float32)


## 4. Operations on laws

The operations form a registry, which lists each operation's operands and routes.

In [22]:
summaries = operation_registry.list()
pd.DataFrame(
    {"operands": [", ".join(operand.name for operand in s.operands) for s in summaries],
     "kind": ["derived" if s.is_derived else "primitive" for s in summaries],
     "routes": [", ".join(route.name for route in s.routes) for s in summaries]},
    index=[s.name for s in summaries],
)

,operands,kind,routes
convert,"d, target",primitive,"identity, converters, normalize"
sample,"d, sample_shape",primitive,"exact, normalize"
condition_on,"d, given",primitive,"curry, slice, exact_conditioning, bayes, approximate_conditioning, unnormalized"
log_prob,"d, value",primitive,exact
unnormalized_log_prob,"d, value",primitive,exact
prob,"d, value",derived,identity
unnormalized_prob,"d, value",derived,identity
random_log_prob,M,primitive,exact
random_unnormalized_log_prob,M,primitive,exact
evaluate,"f, v, fixed_args",primitive,evaluation_rules


`describe` states one operation's contract, with its routes in selection order.

In [23]:
print(operation_registry.describe("mean"))

mean(d) — primitive
  The mean ``E[X]`` for ``X ~ d``, a value shaped like one draw.
  operands:
    d: DistributionSpec; read by the result rule
  routes, in selection order:
    closed_form (capability, exact; requires SupportsMean): membership in SupportsMean suffices
    monte_carlo (fallback, approximate): The law samples numeric draws, which average coordinatewise, or laws, which average to a mixture.


`sample` returns one tracked draw, or a batch on a level named `sample`, and each draw is labeled by the law it comes from.
`with_options(raw=True)` returns the representation instead, and draws inside `workflow_run(seed=...)` are reproducible (Section 12).

In [24]:
mu_prior = Normal("mu", 0.0, 5.0)
with workflow_run(seed=0):
    one = sample(mu_prior)
    many = sample(mu_prior, sample_shape=(5,))
    raw_many = sample.with_options(raw=True)(mu_prior, sample_shape=(5,))
show({"one draw": repr(one), "five draws": repr(many), "values of the five draws": many.values,
      "raw draws": raw_many, "raw draws are a jax.Array": isinstance(raw_many, jax.Array)})

one draw                   NumericArray('mu', shape=(), dtype=float32, support=real)
five draws                 NumericArrayBatch(
                               'mu',
                               levels={'sample': 5},
                               element_spec=NumericArraySpec(shape=(), dtype=float32, support=real),
                           )
values of the five draws   [-5.127493  -1.9288195 -0.6630897  6.3674555  3.6620731]
raw draws                  [-14.860659   -3.721889   -6.036863    3.994918    6.0235777]
raw draws are a jax.Array  True


A joint's draws form a record batch whose fields follow the prior's nested record, and an element is labeled by its position.

In [25]:
with workflow_run(seed=0):
    prior_draws = sample(prior, sample_shape=(3,))
show({"prior_draws": repr(prior_draws), "prior_draws[0]": repr(prior_draws[0])})

prior_draws     NumericRecordBatch(
                    'prior',
                    levels={'sample': 3},
                    fields=('population/mu', 'population/tau', 'groups/theta_tilde'),
                )
prior_draws[0]  NumericRecord('prior[sample=0]', fields=('population/mu', 'population/tau', 'groups/theta_tilde'))


The functionals return values of the event's kind, labeled by the law.
**Transitional:** `cov` returns a dense array (Status).

In [26]:
z = families[-1]
show({"log_prob(z, [0, 0])": repr(log_prob(z, jnp.zeros(2))), "mean(z)": repr(mean(z)),
      "variance(z)": repr(variance(z)), "cov(z)": repr(cov(z)),
      "quantile(z, 0.5)": repr(quantile(z, 0.5))})

log_prob(z, [0, 0])  NumericArray('z', shape=(), dtype=float32)
mean(z)              NumericArray('z', shape=(2,), dtype=float32, support=real)
variance(z)          NumericArray('z', shape=(2,), dtype=float32, support=non_negative)
cov(z)               NumericArray('z', shape=(2, 2), dtype=float32)
quantile(z, 0.5)     NumericArray('z', shape=(2,), dtype=float32)


`quantile` at several levels returns a batch on a level named `quantile`, with the level axis leading, so its values have one row per level.

In [27]:
levels = quantile(z, jnp.array([0.05, 0.5, 0.95]))
show({"levels": repr(levels)})
pd.DataFrame(np.asarray(levels.values), columns=["z[0]", "z[1]"],
             index=pd.Index([0.05, 0.5, 0.95], name="quantile")).round(3)

levels  NumericArrayBatch(
            'z',
            levels={'quantile': 3},
            element_spec=NumericArraySpec(shape=(2,), dtype=float32),
        )


,z[0],z[1]
quantile,,
0.05,-1.645,-2.326
0.50,0.000,0.000
0.95,1.645,2.326


`expectation(d, f)` is `mean(evaluate(f, d))`, here the mean of the sampling lift's 256 draws.

In [28]:
@function
def square(x):
    return x**2


with workflow_run(seed=0):
    second_moment = expectation(mu_prior, square)
show({"expectation(mu_prior, square)": float(second_moment), "exact value": 25.0})

expectation(mu_prior, square)  23.501201629638672
exact value                    25.0


`evaluate(f, d)` returns the pushforward law, which for a nonlinear map is an empirical law over the outputs, labeled by the map.

In [29]:
with workflow_run(seed=0):
    pushforward = evaluate(square, mu_prior)
pushforward

EmpiricalDistribution(
    'square',
    atoms=NumericArrayBatch(
        'square',
        levels={'draw': 256},
        element_spec=NumericArraySpec(shape=(), dtype=float32),
    ),
)

Its provenance records the rule that realized it, and its mean equals the `expectation` above, since both draw under seed 0.

In [30]:
show({"pushforward.provenance.metadata": pushforward.provenance.metadata,
      "mean(pushforward)": float(mean(pushforward))})

pushforward.provenance.metadata  {'func': 'evaluate', 'route': 'evaluation_rules', 'exact': False, 'method': 'sampling_lift'}
mean(pushforward)                23.501201629638672


`check` runs the call up to the selection of a route without executing it, and returns a `CallReport`.
A report displays as a table with one row per route, and the selected route is marked.

In [31]:
mean.check(z)

,route,method,exact,feasible,reason
selected,closed_form,,exact,feasible,


The report's repr reads as the call of its constructor.

In [32]:
print(repr(mean.check(z)))

CallReport(
    routes=(MethodInfo(True, method_name='closed_form', exact=True),),
    selected=MethodInfo(True, method_name='closed_form', exact=True),
    result=OutputSpec(**{'mean(z)': NumericArraySpec(shape=(2,), dtype=float32, support=real)}),
)


`expectation`'s report shows its own registry: the exact method is infeasible for a `Normal`, so the Monte Carlo method is selected.

In [33]:
expectation.check(mu_prior, square)

,route,method,exact,feasible,reason
,closed_form,,exact,infeasible,Normal does not claim SupportsExpectation
,evaluation_rules,exact methods,exact,infeasible,"No feasible method for (Function, Normal) with exact_only. Tried: empirical_enumeration: Normal 'mu' is not an empirical law"
selected,evaluation_rules,sampling_lift,approximate,feasible,
,,,,deferred,the result's type is completed from the returned value


## 5. The hierarchical model three ways

The non-centered eight-schools model has `mu ~ N(0, 5)`, `tau ~ HalfCauchy(0, 5)`, `theta_tilde_j ~ N(0, 1)`, and `y_j ~ N(mu + tau theta_tilde_j, sigma_j)`.
The three representations below are conditioned on the observed effects, and where a program names its parameters flat, a rename nests its posterior into the parameter record `{population: {mu, tau}, groups: {theta_tilde}}`.

### Stan

`StanModel` is a `ConditionalDistribution` from the program's data block to the unnormalized posterior over its parameters.
The adapter reads both declarations from the program before any data are bound.
Each data entry declares its dtype, each parameter declares its support, and `J` stays a symbolic dimension.

In [34]:
STAN_PROGRAM = """
data {
  int<lower=1> J;
  vector[J] y;
  vector<lower=0>[J] sigma;
}
parameters {
  real mu;
  real<lower=0> tau;
  vector[J] theta_tilde;
}
model {
  mu ~ normal(0, 5);
  tau ~ cauchy(0, 5);
  theta_tilde ~ normal(0, 1);
  y ~ normal(mu + tau * theta_tilde, sigma);
}
"""
stan_file = Path(tempfile.mkdtemp()) / "eight_schools.stan"
stan_file.write_text(STAN_PROGRAM)
stan_model = StanModel("eight_schools", str(stan_file))
stan_data = {"J": J, "y": np.asarray(y_obs), "sigma": np.asarray(sigma)}
pd.concat({"given": spec_table(stan_model.given_spec),
           "parameter": spec_table(stan_model.event_spec.spec)})

shape    dtype   support
given     J              ()    int32      None
          y            (J,)  float32      None
          sigma        (J,)  float32      None
parameter mu             ()  float32      real
          tau            ()  float32  positive
          theta_tilde  (J,)  float32      real

Binding the data curries the kernel to its unnormalized posterior, and `condition_on` normalizes it with nutpie, which is the highest-ranked method that applies.

In [35]:
condition_on.check(stan_model, stan_data)

,route,method,exact,feasible,reason
,curry,exact methods,exact,infeasible,"the exact stage's result is unnormalized, and no exact method normalizes it; method=""unnormalized"" returns that result: No method registered for _StanPosterior with exact_only"
,slice,exact methods,exact,infeasible,route 'slice' declined: the conditioned object is not a joint law
,exact_conditioning,,exact,infeasible,StanModel does not claim SupportsExactConditioning
,bayes,exact methods,exact,infeasible,route 'bayes' declined: the given names no produced field
selected,curry,nutpie_nuts,approximate,feasible,


The posterior is an empirical law over the draws, on the levels `chain` and `draw`, and its annotations record the method that ran.
nutpie takes `progress_bar` among its method options, and `False` turns its progress display off.

In [36]:
NUTS = {"num_warmup": 500, "num_results": 500, "num_chains": 2}
NUTPIE = {**NUTS, "progress_bar": False}
with workflow_run(seed=1):
    stan_fit = condition_on.with_options(method_options=NUTPIE)(stan_model, stan_data)
show({"stan_fit": repr(stan_fit), "method": stan_fit.annotations.attrs["method"]})

stan_fit  EmpiricalDistribution(
              'eight_schools',
              atoms=NumericRecordBatch(
                  'posterior',
                  levels={'chain': 2, 'draw': 500},
                  fields=('mu', 'tau', 'theta_tilde'),
              ),
          )
method    nutpie_nuts


Renaming the posterior nests its fields.

In [37]:
stan_posterior = stan_fit.with_path_names(NEST)
stan_posterior

EmpiricalDistribution(
    'eight_schools',
    atoms=NumericRecordBatch(
        'posterior',
        levels={'chain': 2, 'draw': 500},
        fields=('mu', 'tau', 'theta_tilde'),
    ),
    event_spec=OutputSpec(
        NumericRecordSpec(
            population=NumericRecordSpec(
                mu=NumericArraySpec(shape=(), dtype=float32, support=real),
                tau=NumericArraySpec(shape=(), dtype=float32, support=positive),
            ),
            groups=NumericRecordSpec(
                theta_tilde=NumericArraySpec(shape=(8,), dtype=float32, support=real),
            ),
        ),
    ),
)

### PyMC

`PyMCModel` is the joint law that a model-building function defines over its free variables.
An argument that defaults to `None` and feeds an observed variable is an event field, so this model is a `Distribution` over `{mu, tau, theta_tilde, y}`.
Every other argument of the builder would be a given slot, so `eight_schools_pymc(sigma)` returns a builder whose only argument is `y`, and it reads the number of schools from `sigma`.

In [38]:
def eight_schools_pymc(sigma):
    sigma = np.asarray(sigma)

    def build(y=None):
        import pymc as pm

        with pm.Model() as model:
            mu = pm.Normal("mu", 0.0, 5.0)
            tau = pm.HalfCauchy("tau", 5.0)
            theta_tilde = pm.Normal("theta_tilde", 0.0, 1.0, shape=len(sigma))
            pm.Normal("y", mu + tau * theta_tilde, sigma, observed=y)
        return model

    return build


pymc_model = PyMCModel("eight_schools", eight_schools_pymc(sigma))
print("pymc_model:", repr(pymc_model))
spec_table(pymc_model.event_spec.spec)

pymc_model: PyMCModel(
    'eight_schools',
    model_fn=eight_schools_pymc.<locals>.build,
    variables=('mu', 'tau', 'theta_tilde', 'y'),
)


,shape,dtype,support
mu,(),float32,real
tau,(),float32,positive
theta_tilde,"(8,)",float32,real
y,"(8,)",float32,real


Conditioning on `y` is Bayes' rule, normalized with nutpie.

In [39]:
condition_on.check(pymc_model, {"y": np.asarray(y_obs)})

,route,method,exact,feasible,reason
,curry,exact methods,exact,infeasible,route 'curry' declined: the conditioned object is not a kernel
,slice,exact methods,exact,infeasible,route 'slice' declined: the conditioned object is not a joint law
,exact_conditioning,,exact,infeasible,PyMCModel does not claim SupportsExactConditioning
,bayes,exact methods,exact,infeasible,"the exact stage's result is unnormalized, and no exact method normalizes it; method=""unnormalized"" returns that result: No method registered for _UnnormalizedConditional with exact_only"
,curry,approximate methods,approximate,infeasible,route 'curry' declined: the conditioned object is not a kernel
,slice,approximate methods,approximate,infeasible,route 'slice' declined: the conditioned object is not a joint law
,approximate_conditioning,,approximate,infeasible,PyMCModel does not claim SupportsApproximateConditioning
selected,bayes,nutpie_nuts,approximate,feasible,
,,,,deferred,the result's type is completed from the returned value


The fit runs nutpie, and the rename nests the posterior's fields.

In [40]:
with workflow_run(seed=1):
    pymc_fit = condition_on.with_options(method_options=NUTPIE)(pymc_model, {"y": np.asarray(y_obs)})
pymc_posterior = pymc_fit.with_path_names(NEST)
show({"pymc_fit": repr(pymc_fit), "fields of pymc_posterior": list(pymc_posterior.event_spec.spec)})

pymc_fit                  EmpiricalDistribution(
                              'eight_schools | y',
                              atoms=NumericRecordBatch(
                                  'posterior',
                                  levels={'chain': 2, 'draw': 500},
                                  fields=('mu', 'tau', 'theta_tilde'),
                              ),
                          )
fields of pymc_posterior  ['population/mu', 'population/tau', 'groups/theta_tilde']


Renaming a program family before conditioning nests its parameters and keeps its route.

In [41]:
programs = {"Stan": (stan_model, stan_data), "PyMC": (pymc_model, {"y": np.asarray(y_obs)})}
pd.DataFrame(
    {"route of the model": [condition_on.check(model, given).selected.method_name
                            for model, given in programs.values()],
     "route of the renamed model": [condition_on.check(model.with_path_names(NEST), given).selected.method_name
                                    for model, given in programs.values()]},
    index=list(programs),
)

,route of the model,route of the renamed model
Stan,curry/nutpie_nuts,curry/nutpie_nuts
PyMC,bayes/nutpie_nuts,bayes/nutpie_nuts


### A native factored joint

`conditional_distribution` builds a native likelihood from a function of its given values that returns a law.
`noncentered_likelihood(prior, sigma)` builds it for the prior's two groups, which are its given slots, and for the standard errors `sigma`.
Each group arrives as a `NumericRecord`, and the kernel samples and scores through the law the function returns.
Its label describes the model, so its event declaration names the component `y`, which would otherwise default to the label.

In [42]:
def noncentered_likelihood(prior, sigma):
    sigma = jnp.asarray(sigma)

    def law(population, groups):
        return Normal("y", population["mu"] + population["tau"] * groups["theta_tilde"], sigma)

    return conditional_distribution(
        "noncentered_likelihood",
        law,
        given_spec=prior.event_spec.components,
        event_spec=OutputSpec(y=NumericArraySpec(shape=sigma.shape, support=real)),
    )


likelihood = noncentered_likelihood(prior, sigma)
show({"likelihood": repr(likelihood), "given slots": list(likelihood.given_spec),
      "event": likelihood.event_spec.spec})

likelihood   ConditionalDistribution(
                 'noncentered_likelihood',
                 given=('population', 'groups'),
                 event_spec=OutputSpec(y=NumericArraySpec(shape=(8,), dtype=float32, support=real)),
             )
given slots  ['population', 'groups']
event        NumericArraySpec(shape=(8,), dtype=float32, support=real)


Composition is conditional-first: the likelihood conditions on what the prior produces, so `likelihood * prior` is a `FactoredDistribution` over `{y, population, groups}`, labeled by joining the operands' labels with `·`.

In [43]:
schools = likelihood * prior
schools

FactoredDistribution(
    'noncentered_likelihood·prior',
    factors=(
        ConditionalDistribution(
            'noncentered_likelihood',
            given=('population', 'groups'),
            event_spec=OutputSpec(y=NumericArraySpec(shape=(8,), dtype=float32, support=real)),
        ),
        FactoredDistribution(
            'mu·tau',
            factors=(Normal('mu', loc=0.0, scale=5.0), HalfCauchy('tau', loc=0.0, scale=5.0)),
            event_spec=OutputSpec(
                population=NumericRecordSpec(
                    mu=NumericArraySpec(shape=(), dtype=float32, support=real),
                    tau=NumericArraySpec(shape=(), dtype=float32, support=greater_than(0.0)),
                ),
            ),
        ),
        FactoredMultivariateGaussian(
            'theta_tilde',
            factors=(
                Normal('theta_tilde', loc=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], scale=1.0),
            ),
            event_spec=OutputSpec(
                groups=Num

The joint samples ancestrally and scores a draw.

In [44]:
with workflow_run(seed=0):
    joint_draw = sample(schools)
show({"joint_draw": repr(joint_draw),
      "log_prob(schools, joint_draw)": float(log_prob(schools, joint_draw))})

joint_draw                     NumericRecord(
                                   'noncentered_likelihood·prior',
                                   fields=('y', 'population/mu', 'population/tau', 'groups/theta_tilde'),
                               )
log_prob(schools, joint_draw)  -45.360511779785156


Conditioning on `y` resolves to Bayes' rule, and the inference-method registry selects BlackJAX's NUTS.

In [45]:
condition_on.check(schools, {"y": y_obs})

,route,method,exact,feasible,reason
,curry,exact methods,exact,infeasible,route 'curry' declined: the conditioned object is not a kernel
,slice,exact methods,exact,infeasible,"route 'slice' declined: the factor producing ['y'] conditions on ['groups', 'population'], which the given leaves free, so the conditional is not a product of the factors"
,exact_conditioning,,exact,infeasible,FactoredDistribution does not claim SupportsExactConditioning
,bayes,exact methods,exact,infeasible,"the exact stage's result is unnormalized, and no exact method normalizes it; method=""unnormalized"" returns that result: No method registered for _UnnormalizedConditional with exact_only"
,curry,approximate methods,approximate,infeasible,route 'curry' declined: the conditioned object is not a kernel
,slice,approximate methods,approximate,infeasible,"route 'slice' declined: the factor producing ['y'] conditions on ['groups', 'population'], which the given leaves free, so the conditional is not a product of the factors"
,approximate_conditioning,,approximate,infeasible,FactoredDistribution does not claim SupportsApproximateConditioning
selected,bayes,blackjax_nuts,approximate,feasible,
,,,,deferred,the result's type is completed from the returned value


TFP's NUTS runs when `method=` names it, and each posterior's repr names its method.

In [46]:
with workflow_run(seed=1):
    native_posterior = condition_on.with_options(method_options=NUTS)(schools, {"y": y_obs})
    tfp_posterior = condition_on.with_options(method="tfp_nuts", method_options=NUTS)(schools, {"y": y_obs})
show({"native_posterior": repr(native_posterior), "tfp_posterior": repr(tfp_posterior)})

native_posterior  EmpiricalDistribution(
                      'noncentered_likelihood·prior | y',
                      atoms=NumericRecordBatch(
                          'posterior',
                          levels={'chain': 2, 'draw': 500},
                          fields=('population/mu', 'population/tau', 'groups/theta_tilde'),
                      ),
                  )
tfp_posterior     EmpiricalDistribution(
                      'noncentered_likelihood·prior | y',
                      atoms=NumericRecordBatch(
                          'posterior',
                          levels={'chain': 2, 'draw': 500},
                          fields=('population/mu', 'population/tau', 'groups/theta_tilde'),
                      ),
                  )


### Comparison

The school effects `theta_j = mu + tau theta_tilde_j` are a function of three fields of the posterior.
Passing the posterior's field views lifts the function, and the views co-sample, so each evaluation reads one posterior draw.
With `n_broadcast_samples` at the atom count, the lift enumerates every empirical posterior exactly, renamed or not.

In [47]:
@function
def school_effects(mu, tau, theta_tilde):
    return mu + tau * theta_tilde


posteriors = {
    "Stan, nutpie": stan_posterior,
    "PyMC, nutpie": pymc_posterior,
    "native, BlackJAX NUTS": native_posterior,
    "native, TFP NUTS": tfp_posterior,
}
summary_rows = {}
for label, post in posteriors.items():
    with workflow_run(seed=4):
        thetas = school_effects.with_options(n_broadcast_samples=1000)(
            post["population/mu"], post["population/tau"], post["groups/theta_tilde"]
        )
    means, variances = mean(post), variance(post)
    summaries = [
        ("mu", means["mean(population)/mu"], variances["variance(population)/mu"]),
        ("tau", means["mean(population)/tau"], variances["variance(population)/tau"]),
        *[(f"theta[{j}]", mean(thetas).value[j], variance(thetas).value[j]) for j in range(3)],
    ]
    summary_rows[label] = {(name, stat): value for name, m, v in summaries
                           for stat, value in (("mean", float(m)), ("sd", float(v) ** 0.5))}
    summary_rows[label][("lift", "route")] = thetas.provenance.metadata["route"]
pd.DataFrame.from_dict(summary_rows, orient="index").round(2)

mu         tau       theta[0]       theta[1]        \
                       mean    sd  mean    sd     mean    sd     mean    sd   
Stan, nutpie           4.33  3.39  3.60  3.11     6.09  5.63     4.89  4.80   
PyMC, nutpie           4.42  3.32  3.64  3.34     6.41  5.64     4.94  5.02   
native, BlackJAX NUTS  4.47  3.44  3.72  3.19     6.53  5.87     4.80  4.86   
native, TFP NUTS       4.56  3.28  3.40  2.94     5.96  5.10     4.97  4.48   

                      theta[2]                         lift  
                          mean    sd                  route  
Stan, nutpie              3.85  5.23  empirical_enumeration  
PyMC, nutpie              4.04  5.45  empirical_enumeration  
native, BlackJAX NUTS     4.23  5.65  empirical_enumeration  
native, TFP NUTS          4.01  5.35  empirical_enumeration

### Declarations, provenance, and controls

Each posterior declares its target's event, with the Stan dimension `J` bound.

In [48]:
pd.concat({"Stan target": spec_table(stan_model.event_spec.spec),
           "Stan posterior": spec_table(stan_fit.event_spec.spec)}, axis=1)

Stan target                    Stan posterior                   
                  shape    dtype   support          shape    dtype   support
mu                   ()  float32      real             ()  float32      real
tau                  ()  float32  positive             ()  float32  positive
theta_tilde        (J,)  float32      real           (8,)  float32      real

Each posterior's provenance records the route, the method, and whether the step was exact.

In [49]:
runs = {"Stan": stan_fit, "PyMC": pymc_fit, "native, BlackJAX": native_posterior, "native, TFP": tfp_posterior}
pd.DataFrame([run.provenance.metadata for run in runs.values()], index=list(runs))

,func,route,exact,method
Stan,condition_on,curry,False,nutpie_nuts
PyMC,condition_on,bayes,False,nutpie_nuts
"native, BlackJAX",condition_on,bayes,False,blackjax_nuts
"native, TFP",condition_on,bayes,False,tfp_nuts


`with_options` sets an operation's controls, and `options` reports the effective value of each.

In [50]:
strict = condition_on.with_options(exact_only=True)
pd.DataFrame({"condition_on": {name: str(value) for name, value in condition_on.options.items()},
              "with exact_only=True": {name: str(value) for name, value in strict.options.items()}})

,condition_on,with exact_only=True
workflow_kind,WorkflowKind.DEFAULT,WorkflowKind.DEFAULT
n_broadcast_samples,256,256
dispatch,auto,auto
max_workers,None,None
include_inputs,False,False
method,None,None
exact_only,False,True
conversions,{},{}
method_options,{},{}
raw,False,False


`exact_only=True` excludes every approximate route.
No exact method normalizes the unnormalized posterior, so the call raises `ResolutionError`, and the report names `method="unnormalized"`.

In [51]:
try:
    strict(stan_model, stan_data)
except ResolutionError as error:
    print("the call raises:", type(error).__name__)
strict.check(stan_model, stan_data)

the call raises: ResolutionError


,route,method,exact,feasible,reason
,curry,exact methods,exact,infeasible,"the exact stage's result is unnormalized, and no exact method normalizes it; method=""unnormalized"" returns that result: No method registered for _StanPosterior with exact_only"
,slice,exact methods,exact,infeasible,route 'slice' declined: the conditioned object is not a joint law
,exact_conditioning,,exact,infeasible,StanModel does not claim SupportsExactConditioning
,bayes,exact methods,exact,infeasible,route 'bayes' declined: the given names no produced field


`method="unnormalized"` returns the exact stage alone: the unnormalized posterior, which claims only `SupportsUnnormalizedLogProb`.
The two densities differ by the constants that Stan's sampling statements drop.

In [52]:
stan_target = condition_on.with_options(method="unnormalized")(stan_model, stan_data)
native_target = condition_on.with_options(method="unnormalized")(schools, {"y": y_obs})
targets = {
    "Stan": (stan_target, {"mu": 4.0, "tau": 3.0, "theta_tilde": np.zeros(J)}),
    "native": (native_target, {"population": {"mu": 4.0, "tau": 3.0}, "groups": {"theta_tilde": jnp.zeros(J)}}),
}
claims = (SupportsSampling, SupportsLogProb, SupportsUnnormalizedLogProb)
pd.DataFrame(
    {**{p.__name__: [isinstance(target, p) for target, _ in targets.values()] for p in claims},
     "unnormalized_log_prob at mu=4, tau=3": [float(unnormalized_log_prob(target, point))
                                             for target, point in targets.values()]},
    index=list(targets),
)

,SupportsSampling,SupportsLogProb,SupportsUnnormalizedLogProb,"unnormalized_log_prob at mu=4, tau=3"
Stan,False,False,True,-3.390654
native,False,False,True,-42.652267


Conditioning on the upstream field `population` is the exact slice `p(y | population, groups) p(groups)`, since the regrouped prior stays factored and the likelihood curries over `groups`.

In [53]:
population_value = {"mu": 4.0, "tau": 3.0}
condition_on.check(schools, {"population": population_value})

,route,method,exact,feasible,reason
,curry,exact methods,exact,infeasible,route 'curry' declined: the conditioned object is not a kernel
selected,slice,exact methods,exact,feasible,
,,,,deferred,the result's type is completed from the returned value


The slice is a factored joint over `{y, groups}`, whose likelihood has the one given slot `groups` left.

In [54]:
at_population = condition_on(schools, {"population": population_value})
at_population

FactoredDistribution(
    'noncentered_likelihood·theta_tilde',
    factors=(
        ConditionalDistribution(
            'noncentered_likelihood',
            given=('groups',),
            event_spec=OutputSpec(y=NumericArraySpec(shape=(8,), dtype=float32, support=real)),
        ),
        FactoredMultivariateGaussian(
            'theta_tilde',
            factors=(
                Normal('theta_tilde', loc=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], scale=1.0),
            ),
            event_spec=OutputSpec(
                groups=NumericRecordSpec(
                    theta_tilde=NumericArraySpec(shape=(8,), dtype=float32, support=real),
                ),
            ),
        ),
    ),
)

## 6. Field views and marginals

`d[path]` is a field view: a law over the field at `path` that refers to its parent and keeps its parent's label.
A view of a leaf works on the joint and on the posterior.

In [55]:
tau_view = schools["population/tau"]
theta_view = native_posterior["groups/theta_tilde"]
show({"tau_view": repr(tau_view), "event of tau_view": tau_view.event_spec.spec,
      "theta_view": repr(theta_view),
      "posterior mean of theta_tilde": np.asarray(mean(theta_view).value).round(2)})

tau_view                       FieldView('noncentered_likelihood·prior', path='population/tau')
event of tau_view              NumericArraySpec(shape=(), dtype=float32, support=greater_than(0.0))
theta_view                     FieldView('noncentered_likelihood·prior | y', path='groups/theta_tilde')
posterior mean of theta_tilde  [ 0.32  0.08 -0.03  0.06 -0.16 -0.04  0.39  0.05]


A view at a gathering node is a law over the node's record.
At `population`, the view, the detached marginal, and the factor that produces the node are exact, and they agree on the density and the moments.
The view and the marginal keep the joint's label, and the factor keeps its own.

In [56]:
laws = {'schools["population"]': schools["population"],
        'marginal(schools, "population")': marginal(schools, "population"),
        'factor(schools, "population")': factor(schools, "population")}
pd.DataFrame(
    {"label": [law.label for law in laws.values()],
     "log_prob at mu=4, tau=3": [float(log_prob(law, population_value)) for law in laws.values()],
     "log_prob is exact": [log_prob.check(law, population_value).exact for law in laws.values()],
     "mean of mu": [float(mean(law)["mu"]) for law in laws.values()],
     "mean of tau": [float(mean(law)["tau"]) for law in laws.values()]},
    index=list(laws),
)

,label,"log_prob at mu=4, tau=3",log_prob is exact,mean of mu,mean of tau
"schools[""population""]",noncentered_likelihood·prior,-5.216882,True,0.0,inf
"marginal(schools, ""population"")",mu·tau,-5.216882,True,0.0,inf
"factor(schools, ""population"")",mu·tau,-5.216882,True,0.0,inf


A view derives its capabilities from its parent's.
The view of `y` claims `SupportsLogProb`, but its guard rejects the call, since the marginal density of `y` integrates out the fields that `y` conditions on.
The view of `tau` scores, since the factor that produces `tau` conditions on nothing.

In [57]:
y_view = schools["y"]
print("y_view claims SupportsLogProb:", isinstance(y_view, SupportsLogProb))
try:
    log_prob(y_view, y_obs)
except ResolutionError as error:
    print(f"{type(error).__name__}: {error}")
print("log_prob(tau_view, 1.0):", float(log_prob(tau_view, 1.0)))

y_view claims SupportsLogProb: True
ResolutionError: log_prob: no route applies. Tried: exact: the marginal at ['y'] integrates out the fields ['population', 'groups'], which the requested fields condition on
log_prob(tau_view, 1.0): -2.1002414226531982


**Open question:** a public capability for the density of a subset of observations, such as one school's `y_j`.

Sibling views co-sample when lifted together, since the lift groups arguments by their root and draws the parent once per repetition, while detached marginals draw independently.
The standardized residual `(y - mu - tau theta_tilde) / sigma` is standard normal under the joint, so its spread shows whether the fields came from one joint draw.
Two views of one field, built separately, also co-sample.

In [58]:
@function
def residual(y, mu, tau, theta_tilde, sigma):
    return (y - mu - tau * theta_tilde) / sigma


@function
def difference(a, b):
    return a - b


fields = ["population/mu", "population/tau", "groups/theta_tilde"]
with workflow_run(seed=0):
    from_views = residual(schools["y"], *(schools[path] for path in fields), sigma)
    from_marginals = residual(schools["y"], *(marginal(schools, path) for path in fields), sigma)
    same_field = difference(schools["population/mu"], schools["population/mu"])
show({"residual sd, sibling views": round(float(jnp.std(from_views.atoms.values)), 2),
      "residual sd, detached marginals": round(float(jnp.std(from_marginals.atoms.values)), 2),
      "max |a - b|, one field viewed twice": float(jnp.abs(same_field.atoms.values).max())})

residual sd, sibling views           1.02
residual sd, detached marginals      7.03
max |a - b|, one field viewed twice  0.0


`marginal` returns the detached marginal, which keeps its parent's label.
A root field's marginal is its exact factor.

In [59]:
mu_marginal = marginal(schools, "population/mu")
mu_marginal

Normal(
    'noncentered_likelihood·prior',
    loc=0.0,
    scale=5.0,
    event_spec=OutputSpec(mu=NumericArraySpec(shape=(), dtype=float32, support=real)),
)

A field that the likelihood produces has no exact marginal.
**Transitional:** the Monte Carlo route rejects every call (Status), so `check` reports no route and the call raises `ResolutionError`.

In [60]:
try:
    marginal(schools, "y")
except ResolutionError as error:
    print("the call raises:", type(error).__name__)
marginal.check(schools, "y")

the call raises: ResolutionError


,route,method,exact,feasible,reason
,exact,,exact,infeasible,"the marginal at ['y'] integrates out the fields ['population', 'groups'], which the requested fields condition on"
,monte_carlo,,approximate,infeasible,the empirical marginal of projected draws is not implemented


On the empirical posterior every marginal is exact, and so is the marginal of a selection.

In [61]:
pair = marginal(native_posterior, ("groups/theta_tilde", "population/mu"))
pair

EmpiricalDistribution(
    'noncentered_likelihood·prior | y',
    atoms=NumericRecordBatch(
        "posterior[('groups/theta_tilde', 'population/mu')]",
        levels={'chain': 2, 'draw': 500},
        fields=('theta_tilde', 'mu'),
    ),
)

A selection's fields follow the order of its paths, for a view and a marginal alike, on the posterior and on a factored joint.

In [62]:
ab = Normal("a", 0.0, 1.0) * Normal("b", 5.0, 1.0)
posterior_paths, ab_paths = ("groups/theta_tilde", "population/mu"), ("b", "a")
selections = {
    "view of the posterior": (posterior_paths, native_posterior[posterior_paths]),
    "marginal of the posterior": (posterior_paths, pair),
    "view of ab": (ab_paths, ab[ab_paths]),
    "marginal of ab": (ab_paths, marginal(ab, ab_paths)),
}
pd.DataFrame({"paths": [paths for paths, _ in selections.values()],
              "fields": [list(law.event_spec.spec) for _, law in selections.values()]},
             index=list(selections))

,paths,fields
view of the posterior,"(groups/theta_tilde, population/mu)","[theta_tilde, mu]"
marginal of the posterior,"(groups/theta_tilde, population/mu)","[theta_tilde, mu]"
view of ab,"(b, a)","[b, a]"
marginal of ab,"(b, a)","[b, a]"


The view and the marginal of `tau` agree on its mean, which diverges, so both return `inf`.

In [63]:
tau_laws = {"tau_view": tau_view, 'marginal(schools, "population/tau")': marginal(schools, "population/tau")}
pd.DataFrame({"mean": [float(mean(law)) for law in tau_laws.values()],
              "route": [mean.check(law).selected.method_name for law in tau_laws.values()]},
             index=list(tau_laws))

,mean,route
tau_view,inf,closed_form
"marginal(schools, ""population/tau"")",inf,closed_form


`factor` returns the building block that produces a component, under the block's own label: the likelihood for `y` and a packaged sub-joint for each group.

In [64]:
blocks = {name: factor(schools, component_name=name) for name in ("y", "population", "groups")}
pd.DataFrame(
    {"label": [block.label for block in blocks.values()],
     "given slots": [list(getattr(block, "given_spec", ())) for block in blocks.values()],
     "factors": [[part.label for part in getattr(block, "factors", ())] for block in blocks.values()]},
    index=[f"factor(schools, {name!r})" for name in blocks],
)

,label,given slots,factors
"factor(schools, 'y')",noncentered_likelihood,"[population, groups]",[]
"factor(schools, 'population')",mu·tau,[],"[mu, tau]"
"factor(schools, 'groups')",theta_tilde,[],[theta_tilde]


## 7. Functions and lifting

`@function` wraps a plain callable, and a call on values evaluates it and records provenance.
The annotations decide lifting: a parameter that is unannotated or annotated with a value type lifts a law, while a `Distribution` or `Any` parameter receives the law whole.

In [65]:
@function
def effect_of(mu: float, tau: float, z: float) -> float:
    return mu + tau * z


value = effect_of(4.0, 3.0, 0.5)
show({"effect_of(4.0, 3.0, 0.5)": repr(value), "value": float(value),
      "provenance": repr(value.provenance)})

effect_of(4.0, 3.0, 0.5)  NumericArray('effect_of', shape=(), dtype=float32)
value                     5.5
provenance                Provenance('workflow.effect_of', parents=[effect_of])


A law passed to a value parameter takes the sampling lift, whose result is the pushforward: an empirical law over the outputs.
`check` reports the route and the lifted parameters.

In [66]:
effect_of.check(Normal("mu", 0.0, 5.0), 3.0, 0.5)

,route,method,exact,feasible,reason
selected,sampling_lift,,approximate,feasible,
,,,,deferred,the result's type is completed from the returned value


The lifted call returns the empirical law of 256 outputs.

In [67]:
with workflow_run(seed=0):
    lifted = effect_of(Normal("mu", 0.0, 5.0), 3.0, 0.5)
lifted

EmpiricalDistribution(
    'effect_of',
    atoms=NumericArrayBatch(
        'effect_of',
        levels={'draw': 256},
        element_spec=NumericArraySpec(shape=(), dtype=float32),
    ),
)

Its provenance records the route, the sample count, and the lifted arguments.

In [68]:
show(lifted.provenance.metadata)

dispatch        jax
orchestrate     off
n_samples       256
func            effect_of
broadcast_args  ['mu']
route           sampling_lift
exact           False


A batch passed to a value parameter is swept: the function maps over the elements and keeps the batch's level.
`NumericArrayBatch` infers its element spec from the array.

In [69]:
taus = NumericArrayBatch("tau", jnp.array([0.0, 1.0, 5.0, 10.0]), "tau")
swept = effect_of(4.0, taus, 0.5)
show({"taus": repr(taus), "swept": repr(swept), "values of swept": swept.values})

taus             NumericArrayBatch('tau', levels={'tau': 4}, element_spec=NumericArraySpec(shape=(), dtype=float32))
swept            NumericArrayBatch(
                     'effect_of',
                     levels={'tau': 4},
                     element_spec=NumericArraySpec(shape=(), dtype=float32),
                 )
values of swept  [4.  4.5 6.5 9. ]


A law and a batch together give a nested sweep of broadcasts: one pushforward per element, collected in a `DistributionBatch` on the batch's level.

In [70]:
with workflow_run(seed=0):
    nested = effect_of.with_options(n_broadcast_samples=64)(Normal("mu", 0.0, 5.0), taus, 0.5)
nested

DistributionBatch(
    'effect_of',
    levels={'tau': 4},
    element_spec=DistributionSpec(
        event_spec=OutputSpec(effect_of=NumericArraySpec(shape=(), dtype=float32)),
    ),
)

An element is the pushforward at one `tau`, and the batch's mean keeps the level.

In [71]:
show({"label of nested[3]": nested[3].label, "atoms of nested[3]": nested[3].num_atoms,
      "mean(nested)": repr(mean(nested)),
      "values of mean(nested)": np.asarray(mean(nested).values).round(2)})

label of nested[3]      effect_of[tau=3]
atoms of nested[3]      64
mean(nested)            NumericArrayBatch(
                            'effect_of',
                            levels={'tau': 4},
                            element_spec=NumericArraySpec(shape=(), dtype=float32),
                        )
values of mean(nested)  [ 0.07 -0.01  2.56  4.61]


`include_inputs=True` returns the joint empirical law over the lifted inputs and the output, so each atom pairs an input draw with its output.

In [72]:
with workflow_run(seed=0):
    with_inputs = effect_of.with_options(include_inputs=True, n_broadcast_samples=64)(
        Normal("mu", 0.0, 5.0), HalfCauchy("tau", 0.0, 5.0), 0.5
    )
show({"with_inputs": repr(with_inputs)})
spec_table(with_inputs.event_spec.spec)

with_inputs  EmpiricalDistribution(
                 'effect_of',
                 atoms=NumericRecordBatch('effect_of', levels={'draw': 64}, fields=('mu', 'tau', 'effect_of')),
             )


,shape,dtype,support
mu,(),float32,real
tau,(),float32,greater_than(0.0)
effect_of,(),float32,None


An empirical law whose atoms number at most `n_broadcast_samples` is enumerated exactly: each atom is evaluated once and keeps its weight.

In [73]:
tau_atoms = EmpiricalDistribution("tau", jnp.array([0.0, 1.0, 5.0]), weights=jnp.array([0.2, 0.3, 0.5]))
enumerated = effect_of(4.0, tau_atoms, 0.5)
print("route:", enumerated.provenance.metadata["route"],
      "| exact:", enumerated.provenance.metadata["exact"])
pd.DataFrame({"tau": np.asarray(tau_atoms.atoms.values),
              "effect_of": np.asarray(enumerated.atoms.values),
              "weight": np.asarray(enumerated.weights)}).rename_axis("atom").round(3)

route: empirical_enumeration | exact: True


,tau,effect_of,weight
atom,,,
0,0.0,4.0,0.2
1,1.0,4.5,0.3
2,5.0,6.5,0.5


Two empirical laws enumerate together, in either argument order.

In [74]:
z_atoms = EmpiricalDistribution("z", jnp.array([-1.0, 1.0]))
calls = {"effect_of(4.0, tau_atoms, z_atoms)": (4.0, tau_atoms, z_atoms),
         "effect_of(z_atoms, tau_atoms, 0.5)": (z_atoms, tau_atoms, 0.5)}
results = {call: effect_of(*args) for call, args in calls.items()}
pd.DataFrame({"atoms": [result.num_atoms for result in results.values()],
              "route": [result.provenance.metadata["route"] for result in results.values()],
              "exact": [result.provenance.metadata["exact"] for result in results.values()]},
             index=list(results))

,atoms,route,exact
"effect_of(4.0, tau_atoms, z_atoms)",6,empirical_enumeration,True
"effect_of(z_atoms, tau_atoms, 0.5)",6,empirical_enumeration,True


`n_broadcast_samples` sets the number of draws of the sampling lift, and its default is 256, which `set_default_n_broadcast_samples` changes for the session.
With an empirical law and a continuous one, the lift enumerates the atoms and samples the continuous law within each.
The 256 draws become 85 per atom, so the result has 255 atoms, and each draw carries its atom's weight divided by 85.

In [75]:
with workflow_run(seed=0):
    thousand = effect_of.with_options(n_broadcast_samples=1000)(Normal("mu", 0.0, 5.0), 3.0, 0.5)
    mixed = effect_of(Normal("mu", 0.0, 5.0), tau_atoms, 0.5)
show({"atoms at n_broadcast_samples=1000": thousand.num_atoms,
      "atoms of the mixed lift": mixed.num_atoms,
      "route of the mixed lift": mixed.provenance.metadata["route"],
      "distinct weights of the mixed lift": np.unique(np.asarray(mixed.weights)).round(4),
      "atom weights divided by 42": (np.asarray(tau_atoms.weights) / 42).round(4)})

atoms at n_broadcast_samples=1000   1000
atoms of the mixed lift             255
route of the mixed lift             sampling_lift
distinct weights of the mixed lift  [0.0024 0.0035 0.0059]
atom weights divided by 42          [0.0048 0.0071 0.0119]


An `Any` parameter receives the law whole, as a `Distribution` parameter does.

In [76]:
@function
def density_at(law: Any, x: float) -> float:
    return log_prob.with_options(raw=True)(law, x)


density = density_at(Normal("mu", 0.0, 5.0), 1.0)
show({'density_at(Normal("mu", 0.0, 5.0), 1.0)': repr(density), "value": float(density)})

density_at(Normal("mu", 0.0, 5.0), 1.0)  NumericArray('density_at', shape=(), dtype=float32)
value                                    -2.5483765602111816


`BijectorTransformedDistribution` pushes a base law through a bijector, and here the bijector is the canonical map onto the positive half-line.

In [77]:
log_tau = Normal("log_tau", 1.0, 0.5)
tau_lognormal = BijectorTransformedDistribution("tau", log_tau, bijector_for(positive))
tau_lognormal

BijectorTransformedDistribution(
    'tau',
    base=Normal('log_tau', loc=1.0, scale=0.5),
    bijector=Function('exp', parameters=('x',)),
)

A transformed law co-samples with its base, since both share one root, so a function of the two reads one base draw per evaluation.
With another law in place of the base, the two draw independently.

In [78]:
@function
def mismatch(log_t, t):
    return t - jnp.exp(log_t)


with workflow_run(seed=0):
    paired = mismatch(log_tau, tau_lognormal)
    unpaired = mismatch(Normal("other", 1.0, 0.5), tau_lognormal)
show({"max |t - exp(log_t)|, with its base": float(jnp.abs(paired.atoms.values).max()),
      "max |t - exp(log_t)|, with another law": round(float(jnp.abs(unpaired.atoms.values).max()), 2)})

max |t - exp(log_t)|, with its base     0.0
max |t - exp(log_t)|, with another law  11.39


`evaluate(f, d, fixed_args=...)` is the operation form of the call: it selects the same rule, and its report names the rule.

In [79]:
mu_law = Normal("mu", 0.0, 5.0)
fixed = {"tau": 3.0, "z": 0.5}
evaluate.check(effect_of, mu_law, fixed_args=fixed)

,route,method,exact,feasible,reason
,evaluation_rules,exact methods,exact,infeasible,"No feasible method for (Function, Normal) with exact_only. Tried: empirical_enumeration: Normal 'mu' is not an empirical law"
selected,evaluation_rules,sampling_lift,approximate,feasible,
,,,,deferred,the result's type is completed from the returned value


`method=` names a rule, and the result is labeled by the map.
**Transitional:** the registry holds three rules (Status).

In [80]:
with workflow_run(seed=0):
    evaluated = evaluate.with_options(method="sampling_lift")(effect_of, mu_law, fixed_args=fixed)
show({"label": evaluated.label, "atoms": evaluated.num_atoms,
      "provenance": evaluated.provenance.metadata,
      "registered rules": probpipe.evaluation_rule_registry.list_methods()})

label             effect_of
atoms             256
provenance        {'func': 'evaluate', 'route': 'evaluation_rules', 'exact': False, 'method': 'sampling_lift'}
registered rules  ['empirical_enumeration', 'elementwise_sweep', 'sampling_lift']


## 8. Batching

A `DistributionBatch` holds separate laws that share one event declaration.
The component defaults to the label, so the elements share a label as well.

In [81]:
school_laws = DistributionBatch("schools", [Normal("effect", y, s) for y, s in zip(y_obs, sigma)], "school")
school_laws

DistributionBatch(
    'schools',
    levels={'school': 8},
    element_spec=DistributionSpec(
        event_spec=OutputSpec(effect=NumericArraySpec(shape=(), dtype=float32, support=real)),
    ),
)

An element is a view of its stored law, labeled by its position.

In [82]:
school_laws[0]

Normal(
    'schools[school=0]',
    loc=28.0,
    scale=15.0,
    event_spec=OutputSpec(effect=NumericArraySpec(shape=(), dtype=float32, support=real)),
)

Laws whose components differ are refused, and the message names the components.

In [83]:
try:
    DistributionBatch("pair", [Normal("a", 0.0, 1.0), Normal("b", 0.0, 1.0)], "law")
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

TypeError: every element of a DistributionBatch must be a Distribution whose event declaration matches the batch's, and the law at 1 departs from it: the batch declares the components ['a'] and the law ['b']


Operations lift over the batch: `sample` nests its draws inside the laws' level, and moments, densities, and quantiles keep the level.

In [84]:
with workflow_run(seed=0):
    batch_draws = sample(school_laws, sample_shape=(5,))
batch_results = {
    "sample(school_laws, sample_shape=(5,))": batch_draws,
    "mean(school_laws)": mean(school_laws),
    "log_prob(school_laws, 0.0)": log_prob(school_laws, 0.0),
    "quantile(school_laws, [0.05, 0.95])": quantile(school_laws, jnp.array([0.05, 0.95])),
}
pd.DataFrame(
    {"label": [result.label for result in batch_results.values()],
     "levels": [dict(zip(result.level_names, result.batch_shape)) for result in batch_results.values()],
     "element shape": [result.element_spec.shape for result in batch_results.values()]},
    index=list(batch_results),
)

,label,levels,element shape
"sample(school_laws, sample_shape=(5,))",schools,"{'school': 8, 'sample': 5}",()
mean(school_laws),schools,{'school': 8},()
"log_prob(school_laws, 0.0)",schools,{'school': 8},()
"quantile(school_laws, [0.05, 0.95])",schools,"{'school': 8, 'quantile': 2}",()


The table gives each school's mean and 90% interval, one row per element.

In [85]:
intervals = np.asarray(batch_results["quantile(school_laws, [0.05, 0.95])"].values, dtype=float)
pd.DataFrame({"mean": np.asarray(mean(school_laws).values, dtype=float),
              "5%": intervals[:, 0], "95%": intervals[:, 1]},
             index=pd.Index(list("ABCDEFGH"), name="school")).round(1)

,mean,5%,95%
school,,,
A,28.0,3.3,52.7
B,8.0,-8.4,24.4
C,-3.0,-29.3,23.3
D,7.0,-11.1,25.1
E,-1.0,-15.8,13.8
F,1.0,-17.1,19.1
G,18.0,1.6,34.4
H,12.0,-17.6,41.6


Two accesses of one batch element co-sample, as one object passed twice does.

In [86]:
first = school_laws[0]
with workflow_run(seed=0):
    two_accesses = difference(school_laws[0], school_laws[0])
    one_object = difference(first, first)
show({"max |a - b|, two accesses": float(jnp.abs(two_accesses.atoms.values).max()),
      "max |a - b|, one object": float(jnp.abs(one_object.atoms.values).max())})

max |a - b|, two accesses  0.0
max |a - b|, one object    0.0


`condition_on` sweeps a batch of givens, so a record batch of three datasets gives a batch of posteriors on the level `dataset`, here with short chains.

In [87]:
datasets = NumericRecordBatch("data", {"y": jnp.stack([y_obs, 0.5 * y_obs, y_obs + 10.0])}, "dataset")
SHORT = {"num_warmup": 200, "num_results": 200, "num_chains": 1}
with workflow_run(seed=2):
    fits = condition_on.with_options(method_options=SHORT)(schools, datasets)
fits

DistributionBatch(
    'noncentered_likelihood·prior | y',
    levels={'dataset': 3},
    element_spec=DistributionSpec(
        event_spec=OutputSpec(
            NumericRecordSpec(
                population=NumericRecordSpec(
                    mu=NumericArraySpec(shape=(), dtype=float32, support=real),
                    tau=NumericArraySpec(shape=(), dtype=float32, support=greater_than(0.0)),
                ),
                groups=NumericRecordSpec(
                    theta_tilde=NumericArraySpec(shape=(8,), dtype=float32, support=real),
                ),
            ),
        ),
    ),
)

Each element is the posterior of one dataset, and its annotations record the method that ran.

In [88]:
pd.DataFrame(
    {"label": [fits[i].label for i in range(3)],
     "method": [fits[i].annotations.attrs["method"] for i in range(3)],
     "posterior mean of mu": np.asarray(mean(fits)["mean(population)/mu"].values).round(2)},
    index=pd.Index(["y", "0.5 y", "y + 10"], name="dataset"),
)

,label,method,posterior mean of mu
dataset,,,
y,(noncentered_likelihood·prior | y)[dataset=0],blackjax_nuts,4.27
0.5 y,(noncentered_likelihood·prior | y)[dataset=1],blackjax_nuts,2.43
y + 10,(noncentered_likelihood·prior | y)[dataset=2],blackjax_nuts,10.02


A given is checked against the declaration at its path before any method runs.
A mapping that holds the three datasets as one array of shape `(3, 8)` does not conform to `y`'s shape `(8,)`, so `check` reports the call infeasible and the call raises.
Several datasets are conditioned on as a batch of givens, as above.

In [89]:
stacked = {"y": jnp.stack([y_obs, 0.5 * y_obs, y_obs + 10.0])}
print("feasible:", condition_on.check(schools, stacked).feasible)
try:
    condition_on(schools, stacked)
except Exception as error:
    print(f"{type(error).__name__}: {str(error).splitlines()[0][:160]}")

feasible: False
ResolutionError: condition_on: no route applies. Tried: curry (exact methods): the given at 'y' does not conform: 'y' has rank 2, expected rank 1 from shape (8,); slice (exact m


`iterate` folds a step over inputs and returns the visited laws as a batch on a level named `iterate`.
Each step here updates a normal law on `mu` with one school's effect, treating `sigma_j` as known.

In [90]:
def update(law, school):
    effect, se = school
    m, v = float(mean(law)), float(variance(law))
    precision = 1.0 / v + 1.0 / se**2
    return Normal("mu", (m / v + effect / se**2) / precision, precision**-0.5)


steps = iterate(update, Normal("mu", 0.0, 5.0), list(zip(y_obs.tolist(), sigma.tolist())))
steps

DistributionBatch(
    'iterate',
    levels={'iterate': 9},
    element_spec=DistributionSpec(
        event_spec=OutputSpec(mu=NumericArraySpec(shape=(), dtype=float32, support=real)),
    ),
)

Each element is the law after one more school.

In [91]:
pd.DataFrame({"mean of mu": np.asarray(mean(steps).values),
              "sd of mu": np.sqrt(np.asarray(variance(steps).values))},
             index=pd.Index(["prior", *"ABCDEFGH"], name="after school")).round(2)

,mean of mu,sd of mu
after school,,
prior,0.00,5.00
A,2.80,4.74
B,3.76,4.29
C,3.30,4.14
D,3.76,3.87
E,3.02,3.56
F,2.83,3.39
G,4.39,3.21
H,4.62,3.16


## 9. Empirical laws and resampling

An `EmpiricalDistribution` is a weighted set of atoms.
A plain array's atoms lie on one level, named by `level`, which defaults to the component.

In [92]:
effects = EmpiricalDistribution("effect", y_obs, weights=1.0 / sigma**2, level="school")
effects

EmpiricalDistribution(
    'effect',
    atoms=NumericArrayBatch(
        'effect',
        levels={'school': 8},
        element_spec=NumericArraySpec(shape=(), dtype=float32),
    ),
    weights=[0.07369122, 0.16580525, 0.06476768, 0.13702913, 0.20469785, 0.13702913, 0.16580525, 0.05117446],
)

With precision weights, the mean is the pooled estimate of a common effect.

In [93]:
pooled = jnp.sum(y_obs / sigma**2) / jnp.sum(1.0 / sigma**2)
show({"mean(effects)": round(float(mean(effects)), 3), "pooled estimate": round(float(pooled), 3),
      "level names of the atoms": effects.atoms.level_names})

mean(effects)             7.686
pooled estimate           7.686
level names of the atoms  ('school',)


A bootstrap replicate draws `replicate_size` atoms with replacement and keeps the dataset's level, so a statistic written for the data applies to a replicate.

In [94]:
unweighted = EmpiricalDistribution("effect", y_obs, level="school")
replicate = BootstrapReplicateDistribution("replicate", unweighted)
replicate

BootstrapReplicateDistribution(
    'replicate',
    source=EmpiricalDistribution(
        'effect',
        atoms=NumericArrayBatch(
            'effect',
            levels={'school': 8},
            element_spec=NumericArraySpec(shape=(), dtype=float32),
        ),
    ),
    replicate_size=8,
)

A draw is a batch on the dataset's level, and lifting the statistic over the replicate law gives its bootstrap distribution.

In [95]:
@function
def average(effect):
    return jnp.mean(effect)


with workflow_run(seed=0):
    one_replicate = sample(replicate)
    bootstrap_mean = average.with_options(n_broadcast_samples=500)(replicate)
show({"levels of a draw": dict(zip(one_replicate.level_names, one_replicate.batch_shape)),
      "values of a draw": one_replicate.values,
      "atoms of bootstrap_mean": bootstrap_mean.num_atoms,
      "bootstrap sd of the mean": round(float(variance(bootstrap_mean)) ** 0.5, 3)})

levels of a draw          {'school': 8}
values of a draw          [28.  8. 28.  7.  8.  1. 28. -3.]
atoms of bootstrap_mean   500
bootstrap sd of the mean  3.34


`BootstrapDistribution` is the random measure whose draw is the empirical law of one replicate, and its mean is a law.

In [96]:
bootstrap = BootstrapDistribution("bootstrap", unweighted)
bootstrap

BootstrapDistribution(
    'bootstrap',
    source=EmpiricalDistribution(
        'effect',
        atoms=NumericArrayBatch(
            'effect',
            levels={'school': 8},
            element_spec=NumericArraySpec(shape=(), dtype=float32),
        ),
    ),
    replicate_size=8,
)

A draw is an empirical law, and the mean is the source's empirical law, in closed form.

In [97]:
with workflow_run(seed=0):
    one_measure = sample(bootstrap)
mean_measure = mean(bootstrap)
show({"class of a draw": type(one_measure).__name__, "atoms of a draw": one_measure.atoms.values,
      "class of the mean": type(mean_measure).__name__,
      "atoms of the mean": mean_measure.atoms.values,
      "route of the mean": mean_measure.provenance.metadata["route"]})

class of a draw    EmpiricalDistribution
atoms of a draw    [28.  8. 28.  7.  8.  1. 28. -3.]
class of the mean  EmpiricalDistribution
atoms of the mean  [28.  8. -3.  7. -1.  1. 18. 12.]
route of the mean  closed_form


`KDEDistribution` smooths the atoms with a kernel, so it has an exact density.
The bandwidth defaults to Scott's rule, and Silverman's is selected by name.
The variance adds the kernel's spread to the atoms' variance.

In [98]:
kde = KDEDistribution("effect", y_obs)
silverman = KDEDistribution("effect", y_obs, bandwidth="silverman")
show({"kde": repr(kde), "log_prob(kde, 10.0)": round(float(log_prob(kde, 10.0)), 3),
      "variance, Scott's rule": round(float(variance(kde)), 2),
      "variance, Silverman's rule": round(float(variance(silverman)), 2),
      "variance of the atoms": round(float(jnp.var(y_obs)), 2)})

kde                         KDEDistribution(
                                'effect',
                                atoms=[28.0, 8.0, -3.0, 7.0, -1.0, 1.0, 18.0, 12.0],
                                kernel=GaussianKernel,
                            )
log_prob(kde, 10.0)         -3.469
variance, Scott's rule      136.98
variance, Silverman's rule  142.05
variance of the atoms       95.44


The empirical law samples and has exact moments and marginals, and only the KDE has a density.

In [99]:
resampling_laws = {"effects": effects, "replicate": replicate, "bootstrap": bootstrap, "kde": kde}
pd.DataFrame({p.__name__.removeprefix("Supports"): [isinstance(law, p) for law in resampling_laws.values()]
              for p in protocols},
             index=[f"{name} ({type(law).__name__})" for name, law in resampling_laws.items()])

,Sampling,LogProb,Mean,Variance,Covariance,Quantile,Marginals
effects (EmpiricalDistribution),True,False,True,True,True,True,True
replicate (BootstrapReplicateDistribution),True,False,False,False,False,False,False
bootstrap (BootstrapDistribution),True,False,True,False,False,False,False
kde (KDEDistribution),True,True,True,True,True,False,False


## 10. Conversion

`convert(d, target)` changes the representation and keeps the event declaration.
The converter registry selects the method, and the report states whether the conversion is exact.
Here an empirical law of draws is fit by a `Gamma` through moment matching, which is approximate.

In [100]:
with workflow_run(seed=0):
    tau_draws = sample(Gamma("tau", 2.0, 0.5), sample_shape=(400,))
tau_empirical = EmpiricalDistribution("tau", tau_draws)
convert.check(tau_empirical, Gamma)

,route,method,exact,feasible,reason
,identity,,exact,infeasible,a EmpiricalDistribution is not a Gamma as it is
,converters,exact methods,exact,infeasible,"No method registered for (EmpiricalDistribution, Gamma) with exact_only"
,normalize,exact methods,exact,infeasible,route 'normalize' declined: the law is normalized
selected,converters,moment_match,approximate,feasible,


The fit keeps the law's label, and its mean matches the draws'.

In [101]:
fitted = convert(tau_empirical, Gamma)
show({"fitted": repr(fitted), "provenance": fitted.provenance.metadata,
      "mean(fitted)": round(float(mean(fitted)), 3),
      "mean(tau_empirical)": round(float(mean(tau_empirical)), 3)})

fitted               Gamma('tau', concentration=2.2657669, rate=0.60083187)
provenance           {'func': 'convert', 'route': 'converters', 'exact': False, 'method': 'moment_match'}
mean(fitted)         3.771
mean(tau_empirical)  3.771


A family converts to an empirical law by sampling, and a source that is already of the target class is returned by the exact `identity` route.

In [102]:
with workflow_run(seed=0):
    gamma_draws = convert(Gamma("tau", 2.0, 0.5), EmpiricalDistribution)
conversions = {"Gamma to EmpiricalDistribution": gamma_draws,
               "Gamma to Gamma": convert(Gamma("tau", 2.0, 0.5), Gamma)}
pd.DataFrame([{"class": type(result).__name__, **result.provenance.metadata}
              for result in conversions.values()], index=list(conversions)).fillna("")

,class,func,route,exact,method
Gamma to EmpiricalDistribution,EmpiricalDistribution,convert,converters,False,empirical
Gamma to Gamma,Gamma,convert,identity,True,


With `exact_only=True`, the moment match is excluded, and the call raises `ResolutionError`.

In [103]:
try:
    convert.with_options(exact_only=True)(tau_empirical, Gamma)
except ResolutionError as error:
    print(f"{type(error).__name__}: {error}")

ResolutionError: convert: no route applies with exact_only. Tried: identity: a EmpiricalDistribution is not a Gamma as it is; converters (exact methods): No method registered for (EmpiricalDistribution, Gamma) with exact_only; normalize (exact methods): route 'normalize' declined: the law is normalized


A moment-matched fit keeps the source's support.
Its `check` reports a family on another support infeasible before anything is fitted, so counts fit by a `Normal` are refused.

In [104]:
with workflow_run(seed=0):
    counts = EmpiricalDistribution("count", sample(Poisson("count", 3.0), sample_shape=(200,)))
report = converter_registry.check(counts, Normal)
show({"feasible": report.feasible, "reason": report.description})

feasible  False
reason    No feasible method for (EmpiricalDistribution, Normal). Tried: moment_match: Normal is supported on real, and 'count' declares the support non_negative_integer; pass check_support=False to the converter registry to fit it anyway


`convert` therefore raises `ResolutionError` before it fits anything.
The converter registry's `check_support=False` fits the `Normal` anyway, on the real line.

In [105]:
try:
    convert(counts, Normal)
except ResolutionError as error:
    print(f"{type(error).__name__}: {error}")
with workflow_run(seed=0):
    widened = converter_registry.convert(counts, Normal, check_support=False)
show({"widened": repr(widened), "support": str(widened.support)})

ResolutionError: convert: no route applies. Tried: identity: a EmpiricalDistribution is not a Normal as it is; converters (exact methods): No method registered for (EmpiricalDistribution, Normal) with exact_only; normalize (exact methods): route 'normalize' declined: the law is normalized; converters (approximate methods): No feasible method for (EmpiricalDistribution, Normal). Tried: moment_match: Normal is supported on real, and 'count' declares the support non_negative_integer; pass check_support=False to the converter registry to fit it anyway; normalize (approximate methods): route 'normalize' declined: the law is normalized
widened  Normal('count', loc=3.07, scale=1.7677951)
support  real


## 11. A GLM

`glm_likelihood` assembles a likelihood from a response family, its canonical link, and the linear predictor.
**Open question:** the top-level exports of `glm_likelihood` and `GaussianProcess`, so the notebook imports `glm_likelihood` from `probpipe.families`.

In [106]:
rng = np.random.default_rng(0)
n = 40
X = jnp.asarray(np.column_stack([np.ones(n), rng.normal(size=n)]), jnp.float32)
counts_y = jnp.asarray(rng.poisson(np.exp(np.asarray(X) @ np.array([0.5, 0.4]))), jnp.float32)
poisson_likelihood = glm_likelihood("y", PoissonFamily(), X=X)
poisson_likelihood

ConditionalDistribution(
    'y',
    family=PoissonFamily(),
    link=Function('log', parameters=('mean',)),
    X=array(shape=(40, 2), dtype=float32),
    given=('beta',),
)

Binding `X` at construction leaves `beta` as the one given slot, and without `X` both are given slots, with symbolic dimensions.

In [107]:
pd.concat({"X bound": spec_table(poisson_likelihood.given_spec),
           "X given": spec_table(glm_likelihood("y", PoissonFamily()).given_spec)})

shape dtype support
X bound beta             (2,)  None    None
X given X     (obs, features)  None    None
        beta      (features,)  None    None

The likelihood composes with a prior on `beta`, and the default route conditions with BlackJAX's NUTS.

In [108]:
glm_model = poisson_likelihood * MultivariateNormal("beta", jnp.zeros(2), cov=jnp.eye(2))
print("selected route:", condition_on.check(glm_model, {"y": counts_y}).selected.method_name)
with workflow_run(seed=3):
    glm_posterior = condition_on.with_options(method_options=NUTS)(glm_model, {"y": counts_y})
print("glm_posterior:", repr(glm_posterior))
pd.DataFrame({"true value": [0.5, 0.4],
              "posterior mean": np.asarray(mean(glm_posterior)["mean(beta)"].value, dtype=float),
              "posterior sd": np.sqrt(np.asarray(variance(glm_posterior)["variance(beta)"].value, dtype=float))},
             index=["beta[0]", "beta[1]"]).round(3)

selected route: bayes/blackjax_nuts


glm_posterior: EmpiricalDistribution(
    'y·beta | y',
    atoms=NumericRecordBatch('posterior', levels={'chain': 2, 'draw': 500}, fields=('beta',)),
)


,true value,posterior mean,posterior sd
beta[0],0.5,0.660,0.115
beta[1],0.4,0.377,0.142


A posterior predictive is a function of the posterior.
A function that returns an array gives the pushforward of the posterior, and one that returns a law gives a random measure, which here is an empirical law over laws.

In [109]:
X_new = jnp.column_stack([jnp.ones(5), jnp.linspace(-2.0, 2.0, 5)])


@function
def rate(beta, X):
    return jnp.exp(X @ beta)


@function
def predictive(beta, X):
    return Poisson("y_new", jnp.exp(X @ beta))


with workflow_run(seed=0):
    rates = rate.with_options(n_broadcast_samples=200)(glm_posterior["beta"], X_new)
    laws = predictive.with_options(n_broadcast_samples=200)(glm_posterior["beta"], X_new)
    mixture = mean(laws)
laws

EmpiricalDistribution(
    'predictive',
    atoms=DistributionBatch(
        'predictive',
        levels={'draw': 200},
        element_spec=DistributionSpec(
            event_spec=OutputSpec(
                y_new=NumericArraySpec(shape=(5,), dtype=float32, support=non_negative_integer),
            ),
        ),
    ),
)

The mean of the random measure is the predictive mixture, which the Monte Carlo route forms from 256 draws of its laws, since 256 is the default sample count.

In [110]:
mixture

MixtureDistribution(
    'predictive',
    components=256,
    weights=array(shape=(256,), dtype=float32),
    event_spec=OutputSpec(
        y_new=NumericArraySpec(shape=(5,), dtype=float32, support=non_negative_integer),
    ),
)

The mean rate and the predictive mean agree at each new point.
Sampling the mixture is slow (Ergonomics notes), so the table reads the mixture's closed-form mean.

In [111]:
pd.DataFrame({"mean rate": np.asarray(mean(rates).value, dtype=float),
              "predictive mean": np.asarray(mean(mixture).value, dtype=float)},
             index=pd.Index(np.asarray(X_new[:, 1], dtype=float), name="x")).round(3)

,mean rate,predictive mean
x,,
-2.0,0.972,0.948
-1.0,1.366,1.339
0.0,1.958,1.929
1.0,2.863,2.834
2.0,4.266,4.247


**Open questions:** the predictive checks take a sampling kernel, after which `GenerativeLikelihood` retires, and a pointwise predictive density needs a public capability for the density of a subset of observations.

Conditioning on the upstream field `beta` is an exact slice, with no inference.

In [112]:
beta_value = jnp.array([0.5, 0.4])
condition_on.check(glm_model, {"beta": beta_value})

,route,method,exact,feasible,reason
,curry,exact methods,exact,infeasible,route 'curry' declined: the conditioned object is not a kernel
selected,slice,exact methods,exact,feasible,
,,,,deferred,the result's type is completed from the returned value


The slice is the likelihood's law at that `beta`, labeled as the joint.

In [113]:
at_beta = condition_on(glm_model, {"beta": beta_value})
at_beta

Poisson('y', log_rate=array(shape=(40,), dtype=float32))

## 12. Reproducibility

Draws are workflow-owned: `workflow_run(seed=...)` fixes them, a different seed changes them, and a call outside any scope draws fresh values.

In [114]:
def three_draws(law, seed=None):
    if seed is None:
        return np.asarray(sample(law, sample_shape=(3,)).values, dtype=float)
    with workflow_run(seed=seed):
        return np.asarray(sample(law, sample_shape=(3,)).values, dtype=float)


seeds = {"seed 42": 42, "seed 42 again": 42, "seed 43": 43,
         "unscoped": None, "unscoped again": None}
draws = {column: three_draws(mu_prior, seed) for column, seed in seeds.items()}
pd.DataFrame(draws).rename_axis("draw").round(3)

,seed 42,seed 42 again,seed 43,unscoped,unscoped again
draw,,,,,
0,6.831,6.831,5.578,8.989,-2.049
1,3.725,3.725,-3.095,1.604,2.449
2,-7.745,-7.745,0.548,0.308,-0.397


`replay_run(provenance)` re-runs one recorded call on its recorded draws, and it validates the call against the record.
The function must be an importable module-level definition.

In [115]:
@function
def squared(x):
    return x**2


with workflow_run(seed=7):
    recorded = squared(mu_prior)
with replay_run(recorded.provenance):
    replayed = squared(mu_prior)
print("replay reproduces the draws:", bool(jnp.array_equal(recorded.atoms.values, replayed.atoms.values)))

replay reproduces the draws: True


Provenance records the operation, the parents, the call's metadata, and the controls, which hold the randomness record that replay checks.

In [116]:
record = recorded.provenance
show({"operation": record.operation, "parents": [parent.label for parent in record.parents],
      "metadata": record.metadata, "controls": list(record.controls),
      "randomness record": list(record.controls["randomness"])})

operation          broadcast
parents            ['squared', 'mu']
metadata           {'dispatch': 'jax', 'orchestrate': 'off', 'n_samples': 256, 'func': 'squared', 'broadcast_args': ['x'], 'route': 'sampling_lift', 'exact': False}
controls           ['randomness', 'replay']
randomness record  ['schema', 'rng_abi', 'root_words', 'occurrence_path', 'events', 'expected_event_count']


`provenance_ancestors` lists the lineage of a term.

In [117]:
print("ancestors of native_posterior:", [ancestor.label for ancestor in provenance_ancestors(native_posterior)])

ancestors of native_posterior: ['condition_on', 'noncentered_likelihood·prior', 'posterior', 'noncentered_likelihood·prior']


Inference follows the workflow seed as well: one seed gives the same chains, and another seed or a call outside any scope gives new ones.

In [118]:
def posterior_mu(fit, model, given, seed=None):
    if seed is None:
        posterior = fit(model, given)
    else:
        with workflow_run(seed=seed):
            posterior = fit(model, given)
    return np.asarray(posterior.atoms["population/mu"])


args = (condition_on.with_options(method_options=SHORT), schools, {"y": y_obs})
pairs = {"seed 0 and seed 0": (0, 0), "seed 0 and seed 1": (0, 1),
         "two unscoped calls": (None, None)}
show({f"{pair} give equal chains": np.array_equal(posterior_mu(*args, a), posterior_mu(*args, b))
      for pair, (a, b) in pairs.items()})

seed 0 and seed 0 give equal chains   True
seed 0 and seed 1 give equal chains   False
two unscoped calls give equal chains  False


## Ergonomics notes

Each item names the section it refers to.

- **Sampling a mixture is slow** (Section 11). The sampler calls each chosen component's sampler separately (`probpipe/families/_mixture.py:131`), so 100 draws from the 256-component predictive mixture took about 30 seconds in a trial run.
- **The mean of an empirical law over laws samples its atoms** (Section 11). The mean of the empirical law over 200 predictive laws is the Monte Carlo mixture of 256 laws drawn from them, while the weighted mixture of all 200 is exact and as cheap to form.
- **The mixed lift rounds the sample count down** (Section 7). An empirical law of three atoms beside a continuous law turns 256 draws into 85 per atom, so the result has 255 atoms.
- **An inference method's own result keeps the label `posterior`** (Sections 6 and 12). A posterior is labeled by its joint and the conditioned fields, as `schools | y`, while its atoms and the method's result in its lineage are labeled `posterior`, so `provenance_ancestors` lists `posterior` and lists the joint twice.
- **Messages name private classes and misuse an article** (Sections 5 and 10). The exact-only refusals name the private classes `_StanPosterior` and `_UnnormalizedConditional`, and the `identity` route's reason reads "a EmpiricalDistribution is not a Gamma as it is".